# W11: FnB 대시보드

`fnb_sales_sample.csv`를 로드해 3패널(매출/주문/메뉴) 대시보드와 Gemini 코멘트를 생성합니다.


In [ ]:
import io
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for fp in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(fp)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}
    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        result = model.generate_content(prompt)
        return getattr(result, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

from pathlib import Path

candidate = Path("shared/fnb_sales_sample.csv")
if candidate.exists():
    df = pd.read_csv(candidate)
else:
    print("기본 샘플 경로 없음. 대체 샘플 사용")
    df = pd.DataFrame(
        {
            "날짜": pd.date_range("2026-01-01", periods=30, freq="D").astype(str),
            "메뉴명": ["치킨", "피자", "샐러드", "파스타"] * 7 + ["치킨", "피자"],
            "총생산량": np.random.RandomState(0).randint(10, 60, 30),
            "매출": np.random.RandomState(1).randint(80000, 200000, 30)
        }
    )


df["날짜"] = pd.to_datetime(df["날짜"])
menu_daily = df.groupby("메뉴명", as_index=False)["매출"].sum().sort_values("매출", ascending=False)

fig = plt.figure(figsize=(12, 6))
ax1 = plt.subplot(2, 2, 1)
ax2 = plt.subplot(2, 2, 2)
ax3 = plt.subplot(2, 2, 3)

daily = df.groupby("날짜")["매출"].sum().reset_index()
ax1.plot(daily["날짜"], daily["매출"], color="tab:blue")
ax1.set_title("일별 매출")

menu_daily_top = menu_daily.head(8)
ax2.bar(menu_daily_top["메뉴명"], menu_daily_top["매출"])
ax2.set_title("메뉴별 매출 상위")

menu_daily.tail(8).plot(kind="bar", x="메뉴명", y="매출", ax=ax3)
ax3.set_title("메뉴별 매출 하위")
plt.tight_layout()
plt.show()

summary_prompt = f"일별 매출총합: {int(df['매출'].sum())}, 메뉴수: {df['메뉴명'].nunique()}, 주문총량: {int(df['총생산량'].sum())}"
comment = use_gemini(
    "요약 KPI를 기반으로 운영 코멘트를 4문장으로 작성해줘: " + summary_prompt,
    "매출 추세가 안정적으로 상승하고 있으며 메뉴 포트폴리오 재조정 여지가 있습니다. 상위 메뉴 집중과 하위 메뉴 개선으로 수익성을 높일 수 있습니다."
)
print(comment)
